# Manual Inspection — Hallucination Detection Pipeline

Load the pipeline output JSONL, inspect label distributions, cross-tabs, and flag low-confidence judgments.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.join('..'))  # project root

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils.io import load_jsonl

%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 5)

In [ ]:
OUTPUT_PATH = '../data/outputs/pipeline_output.jsonl'

records = load_jsonl(OUTPUT_PATH)
df = pd.json_normalize(records)
print(f'Loaded {len(df)} records')
df.head(2)

## 1. Judge Label Distribution

In [ ]:
label_counts = df['judge_label'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart
label_counts.plot(kind='bar', ax=axes[0], color=['steelblue', 'salmon', 'gray'])
axes[0].set_title('Judge Label Counts')
axes[0].set_xlabel('Label')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

# Pie chart
label_counts.plot(kind='pie', ax=axes[1], autopct='%1.1f%%', startangle=90)
axes[1].set_title('Judge Label Distribution')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

print(label_counts)

## 2. Cross-tab: Judge Label × Ground Truth Label

In [ ]:
crosstab = pd.crosstab(
    df['ground_truth_label'],
    df['judge_label'],
    margins=True,
    margins_name='Total'
)
print('Cross-tab: ground_truth_label (rows) × judge_label (cols)')
crosstab

In [ ]:
# Heatmap (excluding margins)
ct_no_margin = pd.crosstab(df['ground_truth_label'], df['judge_label'])
plt.figure(figsize=(7, 4))
sns.heatmap(ct_no_margin, annot=True, fmt='d', cmap='Blues')
plt.title('Judge Label vs Ground Truth Label')
plt.ylabel('Ground Truth Label')
plt.xlabel('Judge Label')
plt.tight_layout()
plt.show()

## 3. Sortable Example Table

In [ ]:
pd.set_option('display.max_colwidth', 120)

display_cols = [
    'pubid', 'question', 'ground_truth_label',
    'judge_label', 'judge_confidence', 'generated_answer'
]
display_cols = [c for c in display_cols if c in df.columns]

table = df[display_cols].sort_values('judge_confidence')
table

## 4. Low-Confidence Judgments (Flagged for Manual Review)

In [ ]:
LOW_CONF_THRESHOLD = 0.70

if 'judge_confidence' in df.columns:
    low_conf = df[df['judge_confidence'] < LOW_CONF_THRESHOLD].copy()
    print(f'Low-confidence examples (< {LOW_CONF_THRESHOLD}): {len(low_conf)}')

    for _, row in low_conf.iterrows():
        print('=' * 80)
        print(f'PubID: {row.get("pubid", "N/A")}  |  GT: {row.get("ground_truth_label", "N/A")}  |  Judge: {row.get("judge_label", "N/A")}  |  Conf: {row.get("judge_confidence", "N/A")}')
        print(f'Question: {row.get("question", "N/A")}')
        print(f'Generated: {str(row.get("generated_answer", ""))[:400]}')
        print(f'Reasoning: {row.get("judge_reasoning", "N/A")}')
else:
    print('judge_confidence column not found — was --skip_judge used?')

## 5. Uncertainty Features Distribution

In [ ]:
uncertainty_cols = [
    'uncertainty_features.mean_token_prob',
    'uncertainty_features.min_token_prob',
    'uncertainty_features.std_token_prob',
    'uncertainty_features.max_prob_gap',
]
available = [c for c in uncertainty_cols if c in df.columns]

if available:
    fig, axes = plt.subplots(1, len(available), figsize=(4 * len(available), 4))
    if len(available) == 1:
        axes = [axes]
    for ax, col in zip(axes, available):
        df[col].hist(bins=20, ax=ax, edgecolor='black')
        ax.set_title(col.split('.')[-1])
        ax.set_xlabel('Value')
        ax.set_ylabel('Count')
    plt.suptitle('Uncertainty Feature Distributions', y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print('No uncertainty feature columns found.')